# Vector Search

Ordered, categorized, labelled data is generally pretty easy to search. You can use filters and queries to look at the features in the dataset. Unfortunately, most data is unlabelled and unstructured, often in the form of free text. 

If you want to develop a retrieval system (search) against free text, you might try to search for keywords. Keyword search can be effective, but is highly limited by the specificity. 

This is where Vector Embeddings come in. Embeddings are a numerical representation of an input, created by pre-trained statistical models. 



InterSystems IRIS supports a Vector datatype directly alongside any other type of data, and provides fast methods for running vector comparisons, providing a way to provide a fast vector search method. 


In this tutorial, we will see how we can build a basic vector search method based on text information, then move to provide effective search of text from PDF documents. 

## Vector search from text queries 

Create a vector search function against a database, we can break it down into a few steps. 

- Setup
    - Create a table with a VECTOR data type column
    - Embed unstructured text into numerical vectors using an embedding model
    - Add the original text and the vectors into the vector store table
- Search
    - Embed the search query into a vector using the same embedding model
    - Query the data, ordering the dataset by `VECTOR_COSINE` — a measure of the mathematical similarity of the vectors (technically the angle between the vectors)
    - Fetch the top result


### Integration with LLMs

Vector search is very good at improving results from Large Language Models (or agents) often termed Retrieval Augmented Generation. This is a pattern by which a user prompt can be used for a vector search against the database, with the results of the search being found. Nowadays this pattern has been largely replaced by an agentic model where the agent can autonomously call the vector search query. This pattern is nonetheless very powerful because the vector search ensures the agent has the answer from a knowledgebase, but the agent can then digest, simplify or apply the information to the topic at hand.  


 

## Create Vector Store in InterSystems IRIS

To create a vector store in InterSystems IRIS, we use a standard SQL `CREATE TABLE` and define a column with the `VECTOR` datatype. The VECTOR datatype requires two input values: 
- Datatype of the individual values in the vector, normally `DOUBLE` 
- Number of dimensions in the vector.

Each embedding model can have a different number of dimensions, so the second value here depends on the embedding model that you are using. In this notebook, we are going to be using OpenAI's `text-embedding-3-small` model. This has 1536 dimensions. 

You may wish to use a small local model (e.g. from the `sentence-transformers` Python library), which would give a different dimensionality, so remember to tweak this to your model. 


In [1]:
import iris 

connection_args = {
    "hostname":"localhost", 
    "port": 32782,
    "namespace": "FHIRSERVER", 
    "username":"SuperUser", 
    "password": "SYS"
}
conn = iris.connect(**connection_args)
cursor = conn.cursor()

sql_create = """CREATE TABLE Diabetes.VectorStore (
                Source VARCHAR(100),
                Text VARCHAR(10000), 
                Embedding VECTOR(DOUBLE, 1536)
                )"""

## Allows cell to be run multiple times 
cursor.execute("DROP TABLE IF EXISTS Diabetes.VectorStore")

cursor.execute(sql_create)
conn.close()


## Create Embeddings

Now we have the table to store the vectors, we can create the embeddings. Let's start by defining the text we are going to have in our table: 

In [2]:
diabetes_texts = [
    "Managing blood glucose levels is an important part of living with diabetes, helping reduce symptoms and lower the risk of serious long-term health complications.",
    "Insulin helps move sugar from the bloodstream into the body's cells, where it can be converted into energy or stored for use at a later time.",
    "Type 1 diabetes occurs when the immune system attacks insulin-producing cells, leaving the body able to produce very little or no insulin of its own.",
    "People with type 2 diabetes may develop resistance to insulin, meaning their cells respond poorly to the hormone and glucose remains elevated in the bloodstream.",
    "Frequent thirst and urination can be warning signs of high blood sugar, particularly when accompanied by tiredness, blurred vision, or unexplained weight loss.",
    "A balanced diet containing vegetables, fibre, protein, and suitable portions of carbohydrates can help maintain glucose levels within a healthier and more stable range.",
    "Regular physical activity may improve the body's response to insulin, allowing muscles to use glucose more effectively while also supporting cardiovascular health and weight management.",
    "A glucose monitor measures the amount of sugar currently in the blood, providing information that can guide decisions about food, medication, exercise, and insulin doses.",
    "Hypoglycaemia describes a blood sugar level that is too low, which may cause sweating, shaking, confusion, hunger, dizziness, or difficulty concentrating on everyday tasks.",
    "Hyperglycaemia is the medical term for excessively high blood glucose, which can develop when the body has insufficient insulin or cannot use insulin effectively.",
    "Some people control type 2 diabetes through dietary adjustments, increased physical activity, weight management, and oral medication prescribed by a qualified healthcare professional.",
    "Insulin injections may be required when the body cannot regulate glucose effectively, replacing or supplementing the insulin that would normally be produced by the pancreas.",
    "Long-term poor glucose control can damage the eyes, kidneys, nerves, blood vessels, and heart, increasing the likelihood of several serious diabetes-related health complications.",
    "Gestational diabetes develops during pregnancy when hormonal changes affect insulin function, and although it often resolves after birth, continued monitoring may still be recommended.",
    "An HbA1c test estimates average blood glucose over the previous few months, giving a broader view of glucose management than a single daily blood sugar measurement."
]

As mentioned above, we are going to use an OpenAI embedding model. This is faster than using a local model, but requires an OpenAI API key. To create an OpenAI API key, go to https://platform.openai.com/login and create an account. From there, you will need to add some money ($5 minimum) and then mint a key. 

Please note, the code shown in this, and all other tutorials combined, will not cost more than a few cents total. If you wish to do this for free using local models, you can explore using [sentence-transformers](https://www.sbert.net/docs/quickstart.html). The main downside of this approach is that it is slower to run and requires large model downloads. 

Once you have an OpenAI key, save it to a file called `.env` in the root of this repository (copy the template `.env.example`), and load with dotenv (shown below). You can also use it inline (shown commented out) but there is a danger it will leak if so. 

In [3]:
from dotenv import load_dotenv
load_dotenv()

# api_key = "sk-proj-aaabbbccc11122233"

True

We are using the openai Python library, which you can install as follows. 

In [4]:
# pip install openai

After this, you can create embeddings using `OpenAI.embeddings.create()`. Let's do this for each of the lines of text:

In [5]:
from openai import OpenAI

client = OpenAI()

# client = OpenAI(api_key=api_key) # Create client with API key in-line

embeddings = [] 

for text in diabetes_texts: 
    response = client.embeddings.create(
        input=text, model="text-embedding-3-small"
    )
    
    embeddings.append(response.data[0].embedding)


In [6]:
print("These are the first 30 values from the vector embedding of the first text to give you an idea: \n")

print(embeddings[0][:30]) 

print("\n And the total number of dimensions of the embedding is: \n")
print(len(embeddings[0]))

These are the first 30 values from the vector embedding of the first text to give you an idea: 

[-0.0012989044189453125, 0.0293731689453125, 0.044921875, 0.03350830078125, -0.026123046875, 0.01123809814453125, 0.01436614990234375, 0.01058197021484375, 0.0176544189453125, 0.039642333984375, -0.03314208984375, -0.00356292724609375, -0.01611328125, 0.01241302490234375, -0.003627777099609375, -0.005481719970703125, -0.02093505859375, 0.036041259765625, 0.020660400390625, 0.0158843994140625, 0.037811279296875, 0.01959228515625, 0.0163116455078125, 0.0311126708984375, -0.0040283203125, -0.003940582275390625, 0.0175323486328125, -0.0401611328125, 0.028472900390625, -0.06280517578125]

 And the total number of dimensions of the embedding is: 

1536


Great, now we have our text and embeddings, now lets add them into the table we created earlier: 

In [7]:
insert_query = "INSERT INTO Diabetes.VectorStore (Source, Text, Embedding) VALUES (?, ?, TO_VECTOR(?))"

conn = iris.connect(**connection_args)
cursor = conn.cursor()

for i in range(len(diabetes_texts)):
    cursor.execute(insert_query, ["Raw_text", diabetes_texts[i], str(embeddings[i])])


If you would like to check that we have added them, try selecting the top row from the table:

In [8]:
cursor.execute("SELECT Source, Text, Embedding FROM Diabetes.VectorStore WHERE ID=1")
row = list(cursor.fetchone())
print(f"Source: {row[0]}")
print(f"Text: {row[1][:30]+'...'}")
print(f"Embedding: {row[2][:100]}...")


Source: Raw_text
Text: Managing blood glucose levels ...
Embedding: -.0012989044189453125,.0293731689453125,.044921875,.03350830078125,-.026123046875,.01123809814453125...


## Search

Now we have our vector store, we can search data by the meaning of the text, rather than the individual words. All we have to do is embed our search query as a vector, then use SQL to query our vector table ranked by vector similarity (found with `VECTOR_COSINE(vector_column, search_vector)`)

Let's break the SQL Query down. 
```SQL
SELECT TOP 1 /* Get First Result */
Text /* We only want the "Text" column returned */
FROM Diabetes.VectorStore /* Our Vector table */
ORDER BY 
VECTOR_COSINE(Embedding, /* comparison between the Embedding column */ 
    TO_VECTOR(?, DOUBLE))  /* and an input, converted to a vector */ 
DESC /* In descending order as higher vector cosine means closer in meaning */ 
```
There are a couple of easy mistakes to make with syntax here so be careful: 
- The vector embedding needs to be passed to InterSystems IRIS as a string in the SQL query
  - i.e `str(query_embedding)` 
- The SQL Query has to convert the vector string to a vector with `TO_VECTOR(vector_string, DOUBLE)` where double is the datatype of each dimension. 

In [9]:
prompt = "Is exercise good for diabetes?"

response = client.embeddings.create(
        input=prompt, model="text-embedding-3-small"
    )
    
prompt_embedding = response.data[0].embedding


search_query = """SELECT TOP 1 
                Text 
                FROM Diabetes.VectorStore 
                ORDER BY 
                VECTOR_COSINE(Embedding, TO_VECTOR(?, DOUBLE)) 
                DESC"""

cursor.execute(search_query, [str(prompt_embedding)])

print(cursor.fetchall())
conn.close()

(("Regular physical activity may improve the body's response to insulin, allowing muscles to use glucose more effectively while also supporting cardiovascular health and weight management.",),)


Here we can see that the search query has returned a relevant result, without having any words in common. Here we have sent a query about "exercise" and it has returned a result about "physical activity". While this may mean the same thing, a standard keyword search wouldn't naturally return it. This is the power of vector search — similar text can be found without any direct match. 

## Reusable vector search function

Finally, lets wrap this up into a reusable function:


In [10]:
import iris
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

def vector_search_diabetes_text(prompt:str) -> list:
    ''' 
    Performs a vector search against the vector table 
    '''
    
    client = OpenAI()

    response = client.embeddings.create(
        input=prompt, model="text-embedding-3-small"
    )

    connection_args = {"hostname":"localhost", "port":32782, "namespace":"FHIRSERVER", "username":"SuperUser", "password":"SYS"}
    
    conn = iris.connect(**connection_args)
    cursor = conn.cursor()


    sql_query = "SELECT TOP 3 Source, Text FROM Diabetes.VectorStore ORDER BY VECTOR_COSINE(Embedding, TO_VECTOR(?, DOUBLE)) DESC"
    cursor.execute(sql_query, [str(response.data[0].embedding)])

    response = cursor.fetchall()
    

    cursor.close()
    conn.close()

    
    
    return response
    



Let's test it out:

In [11]:
answers = vector_search_diabetes_text("What symptoms might suggest that a person's glucose level is too high?")

for answer in answers: 
    print(answer[1], f'\nSource: {answer[0]}\n\n')

Frequent thirst and urination can be warning signs of high blood sugar, particularly when accompanied by tiredness, blurred vision, or unexplained weight loss. 
Source: Raw_text


Hyperglycaemia is the medical term for excessively high blood glucose, which can develop when the body has insufficient insulin or cannot use insulin effectively. 
Source: Raw_text


Hypoglycaemia describes a blood sugar level that is too low, which may cause sweating, shaking, confusion, hunger, dizziness, or difficulty concentrating on everyday tasks. 
Source: Raw_text




## Using longer texts

Using vector search for small bits of text like this can be useful but it is not a very challenging job, and the examples here are so basic that an LLM could probably produce them without needing to search the database. However, longer texts are able to provide much more value. 

In this directory, we have a folder of scientific journal articles on diabetes, saved within PDF files, downloaded from [MDPI](https://www.mdpi.com/). 

Let's add these to our knowledge base.

We are going to use a basic procedure with `pymupdf`, which easily extracts embedded text from PDF documents. Depending on the document, more complex procedures might be needed (for example if the PDF is derived from a scan, OCR methods may be needed to extract the text).

Vector search loses effectiveness if the text gets too long so long text documents need to be broken down into chunks. Chunk size has a great effect on the success of vector search and should be optimised for the use case. 

Here we are going to use the `RecursiveCharacterTextSplitter` from `langchain-text-splitters` to make chunks of up to 1000 characters, with a 100 character overlap. The splitter tries each separator in order, so it will split between sentences where it can, then between words, rather than cutting words in half. 



In [12]:
# pip install pymupdf langchain-text-splitters

In [13]:
import pymupdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=[". ", " ", ""],  # prefer splitting between sentences, then between words
    keep_separator="end",  # keep the full stop at the end of the sentence it belongs to
)

def get_chunks(doc_path: str):
    with pymupdf.open(doc_path) as doc:
        text = " ".join(page.get_text("text") for page in doc)

    text = " ".join(text.split())  # sanitize: collapse whitespace/newlines into single spaces

    chunks = text_splitter.split_text(text)

    return doc_path, chunks

We can use this function for each of the journal articles we have: 

In [14]:
import os
pdfs = ["../data/papers/"+x for x in os.listdir("../data/papers") if x[-4:]==".pdf"]
print(pdfs)

chunks_by_doc = {} 

for pdf in pdfs:
    doc_path, chunks = get_chunks(pdf)
    chunks_by_doc[doc_path] = chunks

['../data/papers/DiabeticRetinopathyPathophysiologyAndTreatments.pdf', '../data/papers/DiabeticWoundHealingScience.pdf', '../data/papers/GlucoseSensingForDiabetesMonitoring.pdf', '../data/papers/PathophysiologyOfType2DiabetesMellitus.pdf', '../data/papers/Type2DiabetesMellitusReviewOfMultiTargetDrugs.pdf']


Take a look at a couple of these chunks at random: 

In [15]:
print(chunks_by_doc["../data/papers/PathophysiologyOfType2DiabetesMellitus.pdf"][13], "\n\n")

print(chunks_by_doc["../data/papers/DiabeticWoundHealingScience.pdf"][7], "\n\n")


3.3. Pathophysiology Regarding the pathophysiology of the disease, a malfunctioning of the feedback loops between insulin action and insulin secretion results in abnormally high glucose levels in blood [2]. In the case of β-cell dysfunction, insulin secretion is reduced, limiting the body’s capacity to maintain physiological glucose levels. On the other hand, IR contributes to increased glucose production in the liver and decreased glucose uptake both in the muscle, liver and adipose tissue. Even if both processes take place early in the pathogenesis and contribute to the development of the disease, β-cell dysfunction is usually more severe than IR. However, when both β-cell dysfunction and IR are present, hyperglycaemia is ampliﬁed leading to the progression of T2DM [35,36]. Int. J. Mol. Sci. 2020, 21, 6275 4 of 34 4. Mechanisms Leading to T2DM and Pathophysiology 4.1. Insulin Secretion: Physiological and Dysfunctional Mechanisms Leading to T2DM 4.1.1. 


Another mechanism by which hy

Now we have our chunks, lets embed them and save them to our vector store: 

In [16]:
insert_query = "INSERT INTO Diabetes.VectorStore (Source, Text, Embedding) VALUES (?, ?, TO_VECTOR(?))"

conn = iris.connect(**connection_args)
cursor = conn.cursor()

# Iterate over the documents
for doc_path, chunks in chunks_by_doc.items():

    # Iterate over the chunks for each document
    for chunk in chunks:
        response = client.embeddings.create(
            input=chunk, model="text-embedding-3-small"
        )
        embedding = response.data[0].embedding

        # Insert into database
        cursor.execute(insert_query, [doc_path, chunk, str(embedding)])

conn.close()

Now lets try our vector search method again, but this time ask some harder questions: 

In [17]:
prompt = "What glucose concentration ranges are reported for the sweat of healthy patients and diabetic patients?"

output = vector_search_diabetes_text(prompt)

for answer in output: 
    print(answer[1], f'\nSource: {answer[0]}\n\n')

Table 1. Summary of glucose concentrations postprandial and pH values measured in physiological ﬂuids of healthy and diabetic patients. Physiological Fluid Biomarker Concentration for Healthy Patients’ Concentration for Diabetic Patients’ pH Blood Glucose 4.9–6.9 mM [19] 2–40 mM [20,40] 7.35–7.45 [14] Interstitial Fluid Glucose 3.9–6.6 [53] 1.99–22.2 [54] 7.2–7.4 [14] Urine Glucose 2.78–5.55 mM [18] >5.55 mM [18] 4.5–8 [14] Sweat Glucose 0.06–0.11 mM [55] 0.01–1 mM [55] 4.5–7 [56] Saliva Glucose 0.23–0.38 mM [57] 0.55–1.77 mM [57] 6.2–7.6 [58] Ocular Fluid Glucose 0.05–0.5 mM [21] 0.5–5 mM [20,21] 6.5–7.6 [18] Breath Acetone 0.1–2 ppm [59] 0.1–103.7 ppm [59] 7.4–8.1 [60] 2.2. Monitoring Glucose in Alternative Physiological Fluids 2.2.1. Interstitial Fluid Interstitial ﬂuid is the extracellular ﬂuid which surrounds tissue cells. It has signiﬁcant potential for medical diagnostics as it possesses a similar composition of a number of clinically important biomarkers to blood [14,15]. 
Sour

In [18]:
prompt = "Why can combination therapy involving several diabetes drugs cause problems for patients?"

output = vector_search_diabetes_text(prompt)

for answer in output: 
    print(answer[1], f'\nSource: {answer[0]}\n\n')

Unluckily, metformin has several side eﬀects (from mild to serious) that cause lack of adherence and therefore it is the antidiabetic oral therapy with the lowest compliance [29]. Although the existence of plenty of diabetic drugs, other drug monotherapy proved unsuccessful in providing satisfactory managing blood glucose levels and the other comorbidities and therefore therapeutic management is often achieved by combinations therapy with drugs that act with diﬀerent mechanism of actions as illustrated in Figure 2 [30]. However, this strategy may be aﬀected by problems related to the polypharmacological approach, such as several side eﬀects, toxicity and unwanted drug–drug interactions. Nevertheless, the beneﬁt of these combinational therapies is often compromised by low patient compliance, which could be improved by the development of two or more active components co-formulated in a single tablet. 
Source: ../data/papers/Type2DiabetesMellitusReviewOfMultiTargetDrugs.pdf


An alternati

# Summary

Here we have seen how vector search can provide effective retrieval of text, based on the semantic meaning of the text rather than just trying to match keywords. 

This vector search function is revisited in the Agents tutorial. The pattern becomes particularly effective when combined with Large Language models as they can parse the chunk of text(s) quickly and summarise the response, making their response improved and more reliable, while the information is easier to find than ever. 

This is a basic implementation of Vector Search, however depending on the use case, the retrieval could be improved in a number of ways: 
- Providing retrieval for the text before and after the chunk to ensure relevant content is included
- Using a hybrid search that also includes key words
- Using a graph vector search to find related information.
- Better sanitizing or chunking of original information.

One other nice feature of vector search is that it is not just for text! There are multi-modal embedding models which can transform both images and texts to vectors, meaning that you could search images using a text query. 

Vector search is only possible with a database that supports Vector datatypes and operations, which InterSystems IRIS natively does. 